Celda 1 — Markdown: Presentación
# ⚽ Asistente Experto en Tácticas de Fútbol — Demo

Este notebook demuestra el funcionamiento del agente RAG construido con LangGraph y Gemini.
Toda la lógica (ingesta, vectorstore, grafo) está modularizada en el paquete `src/`.

**Requisitos previos:**
1. Tener el archivo `api.env` en la raíz con `GOOGLE_API_KEY`.
2. Haber ejecutado `pip install -r requirements.txt`.

In [1]:
# Celda 2 — Code: Configuración inicial e imports

import os
from dotenv import load_dotenv

# 1. Cargar API Key
load_dotenv("api.env")
if not os.getenv("GOOGLE_API_KEY"):
    raise ValueError("❌ Falta GOOGLE_API_KEY en api.env")

# 2. Importar nuestros módulos
from src.corpus import cargar_todo
from src.vectorstore import get_vectorstore
from src.graph import build_graph, extraer_texto
from langchain_core.messages import HumanMessage

print("✅ Entorno configurado y módulos importados.")

✅ Entorno configurado y módulos importados.


## 1. Base de conocimiento (ChromaDB + Gemini Embeddings)
Cargamos los 123 documentos del corpus híbrido (libro + fichas curadas + scraping).

In [2]:
# Celda 4 — Code: Carga y validación del corpus
print("📚 Cargando corpus...")
docs = cargar_todo()
print(f"Total documentos cargados: {len(docs)}")

📚 Cargando corpus...
Total documentos cargados: 123


In [3]:
# Celda 5 — Code: Inicialización del Vectorstore
# En local, cargará el chroma_db_v4 persistente.
# (En Cloud, app.py hace la descompresión automática).
print("🗄️ Inicializando Vectorstore...")
vs = get_vectorstore()
print("✅ Vectorstore listo.")

🗄️ Inicializando Vectorstore...
✅ Vectorstore listo.


# Celda 6 — Markdown: Agente LangGraph
## 2. Agente LangGraph con Memoria
Compilamos el grafo con el `MemorySaver` para mantener la conversación.

In [4]:
# Celda 7 — Code: Compilación del grafo y visualización
grafo = build_graph(vs, model_name="gemini-3.5-flash-lite")

# Visualización del flujo de trabajo
print("Arquitectura del Agente:")
print(grafo.get_graph().draw_mermaid())


Arquitectura del Agente:
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	reformular(reformular)
	retrieve(retrieve)
	generate(generate)
	__end__([<p>__end__</p>]):::last
	__start__ --> reformular;
	reformular --> retrieve;
	retrieve --> generate;
	generate --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



# Celda 8 — Markdown: Interacción y ejemplos de la rúbrica
## 3. Pruebas de la Rúbrica (Preguntas de Ejemplo)

A continuación, se demuestran las 5 capacidades clave exigidas:
1. RAG Básico (Ficha)
2. Memoria conversacional (Reformulador)
3. Consulta conceptual en el libro (Prompt V5)
4. Anti-alucinación
5. Comparación de tácticas

In [5]:
# Celda 9 — Code: Ejemplo 1 — RAG Básico (Ficha)
config_sesion1 = {"configurable": {"thread_id": "sesion_demo_1"}}

pregunta = "¿Qué es el Catenaccio y qué formación base usaba?"
print(f"👤 Usuario: {pregunta}")

respuesta = grafo.invoke(
    {"messages": [HumanMessage(content=pregunta)], "context": [], "query": ""},
    config=config_sesion1
)
print(f"\n🤖 Agente:\n{extraer_texto(respuesta['messages'][-1])}")

👤 Usuario: ¿Qué es el Catenaccio y qué formación base usaba?
🧭 [reformulador] primer turno → query original
🔍 [retrieval] Pregunta: ¿Qué es el Catenaccio y qué formación base usaba?...
   → 6 docs recuperados:
     · [libro_fragmento] frag_050_051.md
     · [tactica] Catenaccio
     · [libro_fragmento] frag_179_180.md
     · [libro_fragmento] frag_047_049.md
     · [libro_fragmento] frag_210_211.md
     · [formacion] 4-4-1-1
🧠 [generation] prompt construido · 6 docs · contexto=9933 chars


e:\Portafolio\Proyecto_IAGenerativa_Asistente_Tacticas\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



🤖 Agente:
- **Entidad principal**: Catenaccio
- **Tipo**: tactica
- **Época**: 1960s
- **Descripción**: Sistema ultra defensivo italiano caracterizado por la figura del líbero libre detrás de la línea de tres centrales para bloquear ataques rivales y lanzar contraataques rápidos.
- **Fortalezas**: Solidez defensiva extrema, transiciones letales, control del ritmo.
- **Debilidades**: Poca posesión, vulnerable ante equipos con creatividad entre líneas.
- **Entrenador / Equipo icónico**: Helenio Herrera / Inter de Milán (1960–1968)
- **Formación base**: 5-3-2
- **Imagen de referencia**: imagenes/catenaccio.png

- **Explicación Didáctica Ampliada**: El Catenaccio, cuyo nombre significa "cerrojo" en italiano, tiene sus raíces iniciales en el sistema *verrou* creado por el austriaco Karl Rappan en Suiza para compensar las limitaciones físicas de jugadores semiprofesionales. No obstante, encontró su máxima expresión en Italia, donde entrenadores como Nereo Rocco y, de manera muy notable, Hel

In [6]:
# Celda 10 — Code: Ejemplo 2 — Memoria y Reformulación
pregunta2 = "¿Y quién fue su entrenador?" # Anáfora: "su"
print(f"👤 Usuario: {pregunta2}")

respuesta2 = grafo.invoke(
    {"messages": [HumanMessage(content=pregunta2)], "context": [], "query": ""},
    config=config_sesion1 # Usa el MISMO hilo
)
print(f"\n🤖 Agente:\n{extraer_texto(respuesta2['messages'][-1])}")

👤 Usuario: ¿Y quién fue su entrenador?


e:\Portafolio\Proyecto_IAGenerativa_Asistente_Tacticas\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


🧭 [reformulador] '¿Y quién fue su entrenador?' → '¿Y quién fue el entrenador principal o más representativo del Catenaccio?'
🔍 [retrieval] Pregunta: ¿Y quién fue el entrenador principal o más representativo del Catenaccio?...
   → 6 docs recuperados:
     · [libro_fragmento] frag_050_051.md
     · [tactica] Catenaccio
     · [libro_fragmento] frag_179_180.md
     · [libro_fragmento] frag_175_178.md
     · [libro_fragmento] frag_079_080.md
     · [libro_fragmento] frag_214_215.md
🧠 [generation] prompt construido · 6 docs · contexto=11127 chars


e:\Portafolio\Proyecto_IAGenerativa_Asistente_Tacticas\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



🤖 Agente:
Basándonos en la ficha táctica del Catenaccio, el entrenador principal y máximo exponente de este sistema ultradefensivo fue **Helenio Herrera**, quien lideró al Inter de Milán entre 1960 y 1968.

A nivel histórico y táctico, el concepto del Catenaccio tuvo varios precursores y evoluciones:
* **Karl Rappan**: Introdujo el sistema *verrou* en Suiza durante los años 30, sentando las bases del hombre libre o líbero.
* **Gipo Viani**: Introdujo el sistema en Italia con la Salernitana, utilizándolo como una estrategia defensiva para competir con recursos limitados (lo que en la literatura táctica se conoce a menudo como el "derecho del débil").
* **Nereo Rocco**: Refinó la táctica en el AC Milan integrando la estructura defensiva con *tornanti* (extremos que retrocedían) y un líbero, combinando la solidez con principios ofensivos.
* **Helenio Herrera**: Llevó el sistema a su máxima expresión competitiva en el Inter de Milán, combinando una ultra-disciplina táctica, rigor psicológ

In [7]:
# Celda 11 — Code: Ejemplo 3 y 4 — Conceptual y Anti-alucinación
config_sesion2 = {"configurable": {"thread_id": "sesion_demo_2"}}

# 3. Conceptual (libro)
pregunta3 = "¿Qué era el tercer defensa en el fútbol de los 50?"
print(f"👤 Usuario: {pregunta3}")
respuesta3 = grafo.invoke(
    {"messages": [HumanMessage(content=pregunta3)], "context": [], "query": ""},
    config=config_sesion2
)
print(f"\n🤖 Agente:\n{extraer_texto(respuesta3['messages'][-1])}")

print("\n" + "="*50 + "\n")

# 4. Anti-alucinación
pregunta4 = "¿Qué táctica usaba el Boca Juniors de 2020?"
print(f"👤 Usuario: {pregunta4}")
respuesta4 = grafo.invoke(
    {"messages": [HumanMessage(content=pregunta4)], "context": [], "query": ""},
    config=config_sesion2
)
print(f"\n🤖 Agente:\n{extraer_texto(respuesta4['messages'][-1])}")

👤 Usuario: ¿Qué era el tercer defensa en el fútbol de los 50?
🧭 [reformulador] primer turno → query original
🔍 [retrieval] Pregunta: ¿Qué era el tercer defensa en el fútbol de los 50?...
   → 6 docs recuperados:
     · [formacion] 5-3-2
     · [libro_fragmento] frag_020_022.md
     · [libro_fragmento] frag_050_051.md
     · [libro_fragmento] frag_240_244.md
     · [libro_fragmento] frag_136_137.md
     · [tactica] Zona Mista
🧠 [generation] prompt construido · 6 docs · contexto=10436 chars


e:\Portafolio\Proyecto_IAGenerativa_Asistente_Tacticas\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



🤖 Agente:
**El tercer defensa y la evolución táctica tras la regla del fuera de juego**

El concepto del "tercer defensa" surge de manera directa como una respuesta táctica a la modificación de la ley del fuera de juego en 1925, la cual redujo de tres a dos el número de jugadores oponentes necesarios entre un atacante y la línea de fondo para habilitarlo. Este cambio revolucionario alteró por completo el equilibrio entre ataque y defensa, ya que las delanteras rivales disponían de mucha más libertad y espacio para recibir balones en posición legal, desbordando los sistemas tradicionales basados en la pirámide 2-3-5.

Para contrarrestar esta nueva vulnerabilidad, entrenadores pioneros como Herbert Chapman en el Arsenal comenzaron a modificar la estructura defensiva. La solución consistió en retrasar a uno de los centrocampistas —tradicionalmente el mediocentro central— para situarlo permanentemente entre los dos zagueros originales, formando así una línea de tres defensores. Este movim

e:\Portafolio\Proyecto_IAGenerativa_Asistente_Tacticas\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


🧭 [reformulador] '¿Qué táctica usaba el Boca Juniors de 2020?' → '¿Qué táctica usaba el Boca Juniors de 2020?'
🔍 [retrieval] Pregunta: ¿Qué táctica usaba el Boca Juniors de 2020?...
   → 6 docs recuperados:
     · [tactica] Laterales Invertidos
     · [tactica] La Máquina
     · [libro_fragmento] frag_050_051.md
     · [formacion] 4-4-1-1
     · [libro_fragmento] frag_079_080.md
     · [libro_fragmento] frag_236_237.md
🧠 [generation] prompt construido · 6 docs · contexto=8982 chars


e:\Portafolio\Proyecto_IAGenerativa_Asistente_Tacticas\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



🤖 Agente:
Lo siento, esa información no se encuentra en mi base de datos táctica actual.


# Celda 12 — Markdown: Chat Interactivo
## 4. Chat Interactivo (Extra)
Ejecuta esta celda si quieres chatear libremente con el agente. Escribe `/salir` para terminar.

In [8]:
# Celda 13 — Code: Chat interactivo (loop)
from src.chat import chat_interactivo

# Inicia el chat
chat_interactivo(grafo)

🤖 Asistente de Tácticas · sesión: sesion_1793718301
Comandos: /salir · /reset · /historial
🧭 [reformulador] primer turno → query original
🔍 [retrieval] Pregunta: que es el tiki taka...
   → 6 docs recuperados:
     · [tactica] Tiki-Taka
     · [libro_fragmento] frag_234_235.md
     · [libro_fragmento] frag_087_088.md
     · [libro_fragmento] frag_050_051.md
     · [libro_fragmento] frag_011_013.md
     · [libro_fragmento] frag_032_033.md
🧠 [generation] prompt construido · 6 docs · contexto=10658 chars


e:\Portafolio\Proyecto_IAGenerativa_Asistente_Tacticas\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



🤖 **El Tiki-Taka**

El Tiki-Taka es un estilo de juego y una evolución directa del Fútbol Total que emergió con fuerza en la década de los 2000, caracterizado por una posesión extrema del balón, la circulación rápida mediante pases cortos y precisos, y una constante movilidad a través de triangulaciones. Esta filosofía busca someter al rival mediante el control absoluto de la pelota, bajo la premisa de que si el adversario no la tiene, no puede marcar; de este modo, la posesión se convierte simultáneamente en la mejor herramienta tanto defensiva como ofensiva.

Su consagración histórica llegó de la mano de Pep Guardiola en el FC Barcelona (2008–2012) y de la selección nacional de España en ese mismo periodo, alcanzando un éxito internacional sin precedentes. Este sistema se estructuraba habitualmente sobre un dibujo táctico de 4-3-3, destacando tácticamente por la utilización de un "falso nueve" (como Lionel Messi) para descolocar las defensas rivales, así como por una presión asfixia

'sesion_1793718301'